# 📏 Week 15 hands-on: Reliable measurement and statistics

## Learning Objectives
In this notebook, you will learn:
1. **Repeated runs** - see how far a pass rate moves when nothing changes, for a SIMULATED stochastic agent and for Week 11's SIMULATED judge
2. **Intervals and the bootstrap** - put a Wilson interval on a pass rate and a bootstrap interval on any statistic
3. **Paired comparisons** - compare v1 and v2 on the same cases, read the discordant cases, and calculate the cases a 2-point gain needs
4. **Judge and reviewer agreement** - put intervals on kappa and sort reviewer disagreements into ledger facts and rubric questions
5. **Segments and the report** - tell a mix shift from segment drift and write the reliability report

## Where this fits
The theory is in `../theory/week-15-reliable-measurement-and-statistics-theory.md`, and the lab walkthrough with exercises and solutions in `week-15-reliable-measurement-and-statistics-handson.md`. This notebook runs the same code as `lab.py`, one step per cell.

## Prerequisites
- The course `.venv` as the kernel (see `setup.md`). No API key, no cost: nothing here calls a model.
- Week 3 (kappa and the example review sheet), Week 11 (the simulated judge) and Week 12 (paired comparisons by case ID).
- Every pass and fail in this notebook is SIMULATED, and every review label is authored by the course, not by a human reviewer.


---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, loads Week 3's and Week 11's labs by path (unchanged), and points at this week's experiment file and second review sheet.


In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Course root, earlier weeks' labs, this week's data
# ============================================================================
import argparse
import copy
import importlib.util
import json
import math
import random
import sys
from collections import defaultdict
from pathlib import Path
from statistics import NormalDist, mean, pstdev

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

HERE = COURSE / "weeks" / "week-15-reliable-measurement-and-statistics" / "handson"
DATA = HERE.parent / "dataset"
EXPERIMENT = DATA / "stochastic-experiment-v1.json"
SECOND_REVIEWER = DATA / "second-reviewer-sheet-v1.csv"


def load_week_lab(folder: str, name: str):
    """Load an earlier week's lab by path, unchanged (Week 6's pattern)."""
    spec = importlib.util.spec_from_file_location(name, COURSE / "weeks" / folder / "handson" / "lab.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


W3 = load_week_lab("week-03-judges-and-human-calibration", "week03_lab_for_w15")
W11 = load_week_lab("week-11-evaluation-harness-and-ci", "week11_lab_for_w15")
print(f"✅ Environment ready: experiment {json.loads(EXPERIMENT.read_text())['experiment_version']}, "
      f"Week 3 samples {len(W3.read_jsonl(W3.SAMPLES))}, Week 11 tool cases {len(W11.read_cases('tools'))}")


---
## 🎲 Part 1: Repeated Runs

The experiment file gives v1 and v2 a pass probability per slice. `simulate_run` decides each case from two seeded normal draws: one fixed per case, shared by every run and both versions, and one fresh per run. `case_share` (0.7) sets the mix, so a hard case stays hard. v2 is built to be exactly 2.0 points better, through two boundary slices.

Below, both versions run 20 times on the same 100 cases, and Week 11's simulated judge grades tool agent v2's unchanged replies with 20 seeds. `discordant` counts the cases where v1 and v2 disagree in each run pair, and `cases_needed_paired` turns that count into the cases a 2-point gain needs (Part 3 explains both).


In [ ]:
# ============================================================================
# REPEATED RUNS: How far a pass rate moves when nothing changes
# ============================================================================
def load_experiment(path: Path = EXPERIMENT) -> dict:
    exp = json.loads(path.read_text())
    probs = exp["pass_probability"]
    if set(probs["v1"]) != set(probs["v2"]):
        raise ValueError("v1 and v2 must give a pass probability for the same slices")
    if not all(0 < p < 1 for version in probs.values() for p in version.values()):
        raise ValueError("pass probabilities must be strictly between 0 and 1")
    if not 0 <= exp["case_share"] <= 1:
        raise ValueError("case_share must be between 0 and 1")
    return exp


def read_cases(exp: dict) -> list[dict]:
    cases = [json.loads(line) for name in exp["cases"]
             for line in (COURSE / name).read_text().splitlines() if line.strip()]
    missing = {c["slice"] for c in cases} - set(exp["pass_probability"]["v1"])
    if missing:
        raise ValueError(f"no pass probability for slices {sorted(missing)}")
    return [{"case_id": c["case_id"], "slice": c["slice"]} for c in cases]


def simulate_run(cases: list[dict], exp: dict, version: str, run: int) -> dict[str, int]:
    """One SIMULATED run: 1 if the case passed, 0 if it failed.

    A case passes when its score falls below a cut-off set by its slice's pass probability.
    The score mixes two standard normal draws: one fixed per case (shared by every run and
    both versions, so a hard case stays hard) and one fresh per version and run (luck).
    `case_share` sets the mix. Each draw is seeded from its own key, so results never depend
    on the order cases run in, and rerunning reproduces them exactly.
    """
    p, share = exp["pass_probability"][version], exp["case_share"]
    results = {}
    for c in cases:
        case_draw = random.Random(f"{exp['seed']}:case:{c['case_id']}").gauss(0, 1)
        run_draw = random.Random(f"{exp['seed']}:{version}:{run}:{c['case_id']}").gauss(0, 1)
        score = math.sqrt(share) * case_draw + math.sqrt(1 - share) * run_draw
        results[c["case_id"]] = int(score < NormalDist().inv_cdf(p[c["slice"]]))
    return results


def expected_pass_rate(cases: list[dict], exp: dict, version: str) -> float:
    """The rate the simulation builds in: known here, never known for a real agent."""
    p = exp["pass_probability"][version]
    return mean(p[c["slice"]] for c in cases)


def z_value(level: float) -> float:
    """1.96 for 95%: the number of standard errors on each side of the estimate."""
    return NormalDist().inv_cdf(0.5 + level / 2)


def discordant(a: dict[str, int], b: dict[str, int]) -> tuple[int, int]:
    """(cases only B passed, cases only A passed). Cases both pass or both fail say nothing about the change."""
    return sum(b[i] > a[i] for i in a), sum(a[i] > b[i] for i in a)


def cases_needed_paired(effect: float, discordance: float, level: float = 0.95, power: float = 0.80) -> int:
    """Cases for a paired comparison to detect `effect` with the given power.

    Each case contributes +1, 0 or -1. The variance of that per-case difference is the
    discordant share minus effect squared.
    """
    z = z_value(level) + NormalDist().inv_cdf(power)
    return math.ceil(z * z * (discordance - effect ** 2) / effect ** 2)


def repeated_runs(exp: dict) -> dict:
    """Pass counts of every SIMULATED run, the gaps between runs of the same version (an A/A test),
    and how many cases disagree between v1 and v2 in each run pair."""
    cases = read_cases(exp)
    runs = {v: [simulate_run(cases, exp, v, r) for r in range(exp["runs"])] for v in ("v1", "v2")}
    counts = {v: [sum(run.values()) for run in rs] for v, rs in runs.items()}
    diffs = [b - a for a, b in zip(counts["v1"], counts["v2"])]
    disc = [sum(discordant(a, b)) for a, b in zip(runs["v1"], runs["v2"])]
    return {"n": len(cases), "counts": counts, "diffs": diffs, "discordant": disc,
            "cases_needed_at_mean": cases_needed_paired(0.02, mean(disc) / len(cases), exp["bootstrap"]["level"], exp["power"]),
            "aa_gap": {v: max(c) - min(c) for v, c in counts.items()},
            "expected": {v: expected_pass_rate(cases, exp, v) for v in ("v1", "v2")}}


def judge_repeats(seeds: range) -> dict:
    """Week 11's tool agent v2 graded by its SIMULATED judge once per seed. The agent's output never changes."""
    passes = []
    for seed in seeds:
        run = W11.run_suite("tools", "v2", True, judge="simulated", seed=seed, now=lambda: "fixed")
        passes.append(run["summary"]["status"]["pass"])
    return {"n": len(W11.read_cases("tools")), "passes": passes, "spread": max(passes) - min(passes)}


def pct(x: float) -> str:
    return f"{100 * x:.1f}%"


def pts(x: float) -> str:
    return f"{100 * x:+.1f}"


def print_repeat(rep: dict, jr: dict) -> None:
    print(f"SIMULATED agent, {rep['n']} cases, {len(rep['counts']['v1'])} runs per version")
    for v in ("v1", "v2"):
        c = rep["counts"][v]
        print(f"  {v}: passed {' '.join(map(str, c))}")
        print(f"      mean {mean(c):.2f}, sd {pstdev(c):.2f}, range {min(c)}-{max(c)} "
              f"(built-in rate {pct(rep['expected'][v])})")
    print(f"  v2 minus v1, run by run: {' '.join(f'{d:+d}' for d in rep['diffs'])}")
    print(f"  runs where v2 looked worse: {sum(d < 0 for d in rep['diffs'])}, "
          f"no better: {sum(d <= 0 for d in rep['diffs'])}, of {len(rep['diffs'])}")
    print(f"  discordant cases per run pair: {min(rep['discordant'])}-{max(rep['discordant'])}, "
          f"mean {mean(rep['discordant']):.1f}; paired cases for +2 points at the mean: {rep['cases_needed_at_mean']:,}")
    print(f"\nSIMULATED judge on Week 11's tool agent v2 ({jr['n']} cases, agent output identical every time)")
    print(f"  passed per seed 0-{len(jr['passes']) - 1}: {' '.join(map(str, jr['passes']))}")
    print(f"  spread {jr['spread']} cases = {100 * jr['spread'] / jr['n']:.1f} points with nothing changed")


exp = load_experiment()
print_repeat(repeated_runs(exp), judge_repeats(range(20)))


v1 alone passes anywhere from 74 to 86 of 100. In 6 of 20 run pairs v2 scores lower than v1, although v2 is built to be better. Between v1 and v2, 8 to 24 cases disagree per run pair, about 16 on average: at that average, detecting a 2-point gain needs 3,142 paired cases. The judge's spread on identical replies is 6 of 30 cases, 20 points. The built-in rates (81.1% and 83.1%) are rates over the slices' probabilities: the 20-run means sit lower because these 100 cases drew slightly harder difficulties.


---
## 📐 Part 2: Intervals and the Bootstrap

**The bootstrap, before the code.** The ideal way to learn how much a pass rate varies is to draw many fresh sets of 100 cases from real traffic and score each one. The bootstrap uses the one set you have as a stand-in for that traffic:

1. Take the 100 case IDs and their results.
2. Draw 100 IDs at random **with replacement**, so some cases appear twice and some not at all.
3. Compute the statistic on the drawn cases.
4. Repeat 10,000 times with a fixed seed, so the result is reproducible.
5. Sort the 10,000 values and keep the middle 95%: the percentile interval.

`bootstrap` does steps 2 to 4 for any statistic, and `percentile_interval` does step 5. For a single pass rate there is also a formula, the Wilson interval, which stays inside 0 to 100% and is never zero-width.


In [ ]:
# ============================================================================
# INTERVALS: Wilson for a pass rate, the bootstrap for anything
# ============================================================================
def wilson(k: int, n: int, level: float = 0.95) -> tuple[float, float]:
    """Wilson score interval for k passes in n cases. Stays inside 0..1 and is never zero-width."""
    if n == 0:
        return 0.0, 1.0
    z, p = z_value(level), k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, centre - half), min(1.0, centre + half)


def percentile_interval(values: list[float], level: float = 0.95) -> tuple[float, float]:
    """The middle `level` of the values: for 95%, the values 2.5% and 97.5% of the way up the sorted list."""
    ordered, tail = sorted(values), (1 - level) / 2
    last = len(ordered) - 1
    return ordered[round(tail * last)], ordered[round((1 - tail) * last)]


def bootstrap(ids: list[str], stat, reps: int, seed: int) -> list[float]:
    """Resample the case IDs with replacement `reps` times and compute `stat` on each resample."""
    rng = random.Random(seed)
    return [stat(rng.choices(ids, k=len(ids))) for _ in range(reps)]


cases = read_cases(exp)
v1_release = simulate_run(cases, exp, "v1", exp["release_run"])
v2_release = simulate_run(cases, exp, "v2", exp["release_run"])
for name, k, n in (("v1 release run", sum(v1_release.values()), 100), ("v2 release run", sum(v2_release.values()), 100),
                   ("30 of 30", 30, 30)):
    lo, hi = wilson(k, n)
    print(f"📐 {name:<15} {k}/{n}  Wilson {pct(lo)} to {pct(hi)}")


def v1_pass_rate(sample):
    return sum(v1_release[i] for i in sample) / len(sample)


lo, hi = percentile_interval(bootstrap(sorted(v1_release), v1_pass_rate, 10000, 15))
print(f"🔁 v1 pass rate, bootstrap of 10,000 resamples: {pct(lo)} to {pct(hi)}")


The bootstrap interval for v1's 81 of 100 (73% to 88%) lands close to the Wilson interval (72.2% to 87.5%): two routes to the same uncertainty. 30 of 30 still leaves room for a failure rate up to about 11%. The two versions' intervals overlap almost entirely, but overlap is the wrong test for two versions run on the same cases. Part 3 compares them directly.


---
## 🔗 Part 3: Paired Comparisons

`paired_difference` resamples case IDs and carries both versions' results with each one. `unpaired_difference` throws the pairing away, for contrast. The discordant cases from Part 1 are the only cases that carry evidence, and `mcnemar_exact` tests their split as coin flips. `cases_needed_paired` (also from Part 1) turns the discordant share into the cases a given effect needs, and `cases_needed_unpaired` does the same for two unrelated samples.


In [ ]:
# ============================================================================
# PAIRED COMPARISON: The release run, then twenty runs averaged per case
# ============================================================================
def paired_difference(a: dict[str, float], b: dict[str, float], reps: int, seed: int,
                      level: float = 0.95) -> dict:
    """B minus A on the same cases. Resample case IDs and carry both versions' results with each."""
    if set(a) != set(b):
        raise ValueError(f"not paired: {len(set(a) ^ set(b))} case IDs are in one version only")
    ids = sorted(a)
    diff = {i: b[i] - a[i] for i in ids}

    def mean_diff(sample: list[str]) -> float:
        # Rounded so that float error (1e-18 instead of 0) cannot move an interval end across zero.
        return round(sum(diff[i] for i in sample) / len(sample), 12)

    lo, hi = percentile_interval(bootstrap(ids, mean_diff, reps, seed), level)
    return {"n": len(ids), "a": mean(a.values()), "b": mean(b.values()), "diff": mean_diff(ids), "lo": lo, "hi": hi}


def unpaired_difference(a: dict[str, float], b: dict[str, float], reps: int, seed: int,
                        level: float = 0.95) -> dict:
    """The same difference with the pairing thrown away: each version's cases resampled separately."""
    rng, ids_a, ids_b = random.Random(seed), sorted(a), sorted(b)
    dist = [round(mean(b[i] for i in rng.choices(ids_b, k=len(ids_b)))
                  - mean(a[i] for i in rng.choices(ids_a, k=len(ids_a))), 12) for _ in range(reps)]
    lo, hi = percentile_interval(dist, level)
    return {"n": len(ids_a), "diff": round(mean(b.values()) - mean(a.values()), 12), "lo": lo, "hi": hi}


def mcnemar_exact(b_only: int, a_only: int) -> float:
    """Two-sided exact McNemar test: if the versions were equal, each discordant case is a coin flip."""
    n = b_only + a_only
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, k) for k in range(max(b_only, a_only), n + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def verdict(lo: float, hi: float) -> str:
    if lo > 0:
        return "IMPROVED"
    if hi < 0:
        return "REGRESSED"
    return "INCONCLUSIVE"


def cases_needed_unpaired(p_a: float, p_b: float, level: float = 0.95, power: float = 0.80) -> int:
    """Cases per version for two independent samples to detect p_b - p_a."""
    z = z_value(level) + NormalDist().inv_cdf(power)
    return math.ceil(z * z * (p_a * (1 - p_a) + p_b * (1 - p_b)) / (p_b - p_a) ** 2)


def per_case_mean(runs: list[dict[str, int]]) -> dict[str, float]:
    """Average each case over repeated runs. The case stays the unit that gets resampled."""
    return {i: mean(run[i] for run in runs) for i in runs[0]}


def compare(exp: dict, runs: int = 1, first_run: int | None = None) -> dict:
    """Compare v1 and v2 on the same 100 cases, from one run or the average of several."""
    cases, boot = read_cases(exp), exp["bootstrap"]
    start = exp["release_run"] if first_run is None else first_run
    a_runs = [simulate_run(cases, exp, "v1", r) for r in range(start, start + runs)]
    b_runs = [simulate_run(cases, exp, "v2", r) for r in range(start, start + runs)]
    a, b = per_case_mean(a_runs), per_case_mean(b_runs)
    paired = paired_difference(a, b, boot["reps"], boot["seed"], boot["level"])
    result = {"runs": runs, "first_run": start, "paired": paired, "verdict": verdict(paired["lo"], paired["hi"]),
              "unpaired": unpaired_difference(a, b, boot["reps"], boot["seed"], boot["level"]),
              "true_diff": expected_pass_rate(cases, exp, "v2") - expected_pass_rate(cases, exp, "v1")}
    if runs == 1:
        b_only, a_only = discordant(a_runs[0], b_runs[0])
        n = len(cases)
        result.update(b_only=b_only, a_only=a_only, mcnemar_p=mcnemar_exact(b_only, a_only),
                      wilson_a=wilson(sum(a_runs[0].values()), n, boot["level"]),
                      wilson_b=wilson(sum(b_runs[0].values()), n, boot["level"]),
                      cases_needed_paired=cases_needed_paired(0.02, (b_only + a_only) / n, boot["level"], exp["power"]),
                      cases_needed_unpaired=cases_needed_unpaired(paired["a"], paired["a"] + 0.02, boot["level"], exp["power"]))
    return result


def print_compare(c: dict) -> None:
    p, u = c["paired"], c["unpaired"]
    label = f"run {c['first_run']}" if c["runs"] == 1 else f"runs {c['first_run']}-{c['first_run'] + c['runs'] - 1}, averaged per case"
    print(f"SIMULATED v1 vs v2 on the same {p['n']} cases, {label}")
    if c["runs"] == 1:
        print(f"  v1 {pct(p['a'])}  Wilson {pct(c['wilson_a'][0])} to {pct(c['wilson_a'][1])}")
        print(f"  v2 {pct(p['b'])}  Wilson {pct(c['wilson_b'][0])} to {pct(c['wilson_b'][1])}")
        print(f"  discordant: {c['b_only']} only v2 passed, {c['a_only']} only v1 passed; "
              f"exact McNemar p = {c['mcnemar_p']:.3f}")
    else:
        print(f"  v1 {pct(p['a'])}  v2 {pct(p['b'])}")
    print(f"  paired bootstrap   v2 minus v1 {pts(p['diff'])} points, 95% interval {pts(p['lo'])} to {pts(p['hi'])}")
    print(f"  unpaired bootstrap v2 minus v1 {pts(u['diff'])} points, 95% interval {pts(u['lo'])} to {pts(u['hi'])}")
    print(f"  verdict: {c['verdict']}  (built-in difference {pts(c['true_diff'])} points)")
    if c["runs"] == 1:
        print(f"  cases needed to detect +2 points with 80% power: paired {c['cases_needed_paired']:,}, "
              f"unpaired {c['cases_needed_unpaired']:,} per version")


print_compare(compare(exp))
print()
print_compare(compare(exp, runs=20, first_run=0))


On the release run v2 leads by 2 points, built from 5 cases only v2 passed against 3 only v1 passed. The paired interval, −3.0 to +8.0, includes zero: INCONCLUSIVE. Without pairing it doubles to −9.0 to +12.0. At this run's 8 discordant cases a 2-point gain needs 1,562 paired cases; at Part 1's average of about 16 it needs 3,142. This set has 100.

Averaging 20 runs per case narrows the paired interval to 0.0 to +4.2, which still touches zero. Repeats remove run-to-run luck. What remains is case sampling, which only more cases reduce.


---
## ⚖️ Part 4: Judge and Reviewer Agreement

`load_verdicts` reads three raters for Week 3's 30 replies: the judge's mock responses (invalid ones dropped), reviewer A (Week 3's example sheet) and reviewer B (this week's second sheet). Both reviewer sheets are authored by the course and labelled `example-not-human-review`. `kappa_interval` bootstraps Week 3's kappa over the case IDs both raters labelled.


In [ ]:
# ============================================================================
# AGREEMENT: Kappa with intervals, and where the reviewers split
# ============================================================================
def load_verdicts() -> dict[str, dict[str, str]]:
    """pass/fail verdicts from the judge (Week 3's mock responses, invalid ones dropped) and two authored reviewers."""
    samples = W3.read_jsonl(W3.SAMPLES)
    judged = W3.build_report(samples, W3.read_jsonl(W3.MOCK_RESPONSES), {}, W3.policy_text())["judgments"]
    return {"judge": {i: j["verdict"] for i, j in judged.items()},
            "reviewer A": {i: r["human_verdict"] for i, r in W3.load_labels(W3.EXAMPLE_LABELS).items()},
            "reviewer B": {i: r["human_verdict"] for i, r in W3.load_labels(SECOND_REVIEWER).items()}}


def kappa_interval(x: dict[str, str], y: dict[str, str], reps: int, seed: int, level: float = 0.95) -> dict:
    """Cohen's kappa (Week 3's formula) on the cases both raters labelled, with a bootstrap interval."""
    ids = sorted(x.keys() & y.keys())

    def kappa(sample: list[str]) -> float:
        return W3.agreement([(x[i], y[i]) for i in sample])["kappa"]

    lo, hi = percentile_interval(bootstrap(ids, kappa, reps, seed), level)
    return {"n": len(ids), "agree": sum(x[i] == y[i] for i in ids), "kappa": kappa(ids), "lo": lo, "hi": hi,
            "disagree": [i for i in ids if x[i] != y[i]]}


def agreement_report(exp: dict) -> dict:
    boot, verdicts = exp["bootstrap"], load_verdicts()
    pairs = [("reviewer A", "reviewer B"), ("judge", "reviewer A"), ("judge", "reviewer B")]
    rows = {f"{x} vs {y}": kappa_interval(verdicts[x], verdicts[y], boot["reps"], boot["seed"], boot["level"])
            for x, y in pairs}
    samples = {s["case_id"]: s for s in W3.read_jsonl(W3.SAMPLES)}
    split = rows["reviewer A vs reviewer B"]["disagree"]
    return {"rows": rows,
            "settled_by_ledger": [i for i in split if W3.forbidden_call(samples[i])],
            "for_rubric_owner": [i for i in split if not W3.forbidden_call(samples[i])]}


def print_agreement(ag: dict) -> None:
    print("AUTHORED labels (example-not-human-review): these numbers test the method, not the judge")
    print(f"{'raters':<26}{'cases':>6}{'agree':>7}{'kappa':>8}   95% interval")
    for name, row in ag["rows"].items():
        print(f"{name:<26}{row['n']:>6}{row['agree']:>7}{row['kappa']:>8.3f}   {row['lo']:.3f} to {row['hi']:.3f}")
    print(f"reviewers disagree on: {', '.join(ag['rows']['reviewer A vs reviewer B']['disagree'])}")
    print(f"  settled by the ledger (forbidden tool call): {', '.join(ag['settled_by_ledger'])}")
    print(f"  for the rubric owner: {', '.join(ag['for_rubric_owner'])}")


print_agreement(agreement_report(exp))


The judge-versus-A kappa of 0.505 comes with an interval from below zero to 0.885: 26 labels cannot tell a good judge from chance. The two reviewers agree at only 0.455, the ceiling for any judge. Three of their six splits are ledger facts (refunds to orders the customer does not own). The other three are rubric ambiguity, and they are also three of the judge's four disagreements with reviewer A.


---
## 🧩 Part 5: Segments and the Report

`slice_table` puts a Wilson interval on each slice. `reweight` computes the overall rate a set of slice rates gives under another mix, using Week 13's SIMULATED baseline and clearance-sale traffic. `build_report` gathers every part into `reliability-report.md` and `reliability-report.json`.


In [ ]:
# ============================================================================
# SEGMENTS AND REPORT: Slice intervals, the mix effect, the reliability report
# ============================================================================
def slice_table(cases: list[dict], a: dict[str, int], b: dict[str, int], level: float = 0.95) -> list[dict]:
    by_slice = defaultdict(list)
    for c in cases:
        by_slice[c["slice"]].append(c["case_id"])
    rows = []
    for name, ids in sorted(by_slice.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        ka, kb = sum(a[i] for i in ids), sum(b[i] for i in ids)
        rows.append({"slice": name, "n": len(ids), "a": ka, "b": kb,
                     "wilson_a": wilson(ka, len(ids), level), "wilson_b": wilson(kb, len(ids), level)})
    return rows


def production_mixes(exp: dict) -> dict[str, dict[str, float]]:
    """Week 13's SIMULATED topic mixes, renamed to this week's slices."""
    mix = exp["production_mix"]
    traffic = json.loads((COURSE / mix["source"]).read_text())
    return {name: {mix["topic_to_slice"][t]: w for t, w in topics.items()}
            for name, topics in traffic["topic_mix"].items()}


def reweight(rates: dict[str, float], mix: dict[str, float]) -> float:
    """The overall pass rate traffic with this slice mix would see, if each slice kept its own rate."""
    return sum(w * rates[s] for s, w in mix.items()) / sum(mix.values())


def segment_report(exp: dict) -> dict:
    cases = read_cases(exp)
    a = simulate_run(cases, exp, "v1", exp["release_run"])
    b = simulate_run(cases, exp, "v2", exp["release_run"])
    runs = [simulate_run(cases, exp, "v1", r) for r in range(exp["runs"])]
    by_slice = defaultdict(list)
    for c in cases:
        by_slice[c["slice"]].append(c["case_id"])
    rates = {s: mean(run[i] for run in runs for i in ids) for s, ids in by_slice.items()}
    mixes = production_mixes(exp)
    covered = set(next(iter(mixes.values())))
    eval_mix = {s: len(by_slice[s]) for s in covered}
    return {"slices": slice_table(cases, a, b, exp["bootstrap"]["level"]), "rates_v1": rates,
            "overall": {"eval set": reweight(rates, eval_mix),
                        **{f"{name} traffic": reweight(rates, m) for name, m in mixes.items()}},
            "mixes": {"eval set": {s: n / sum(eval_mix.values()) for s, n in eval_mix.items()}, **mixes}}


def print_segments(seg: dict) -> None:
    print("SIMULATED release run, by slice")
    print(f"{'slice':<26}{'cases':>6}{'v1':>4}{'v2':>4}   v1 interval        v2 interval")
    for s in seg["slices"]:
        print(f"{s['slice']:<26}{s['n']:>6}{s['a']:>4}{s['b']:>4}   "
              f"{pct(s['wilson_a'][0]):>6} to {pct(s['wilson_a'][1]):<7} {pct(s['wilson_b'][0]):>6} to {pct(s['wilson_b'][1])}")
    print("\nv1 overall on seven slices, each slice at its 20-run rate:")
    for name, rate in seg["overall"].items():
        print(f"  {name:<26}{pct(rate)}")


def build_report(exp: dict) -> dict:
    return {"report_version": "reliability-v1", "experiment_version": exp["experiment_version"],
            "label": exp["label"], "seed": exp["seed"], "bootstrap": exp["bootstrap"],
            "release_run": compare(exp), "twenty_runs": compare(exp, runs=exp["runs"], first_run=0),
            "repeated_runs": repeated_runs(exp), "judge_repeats": judge_repeats(range(20)),
            "agreement": agreement_report(exp), "segments": segment_report(exp)}


DECISION_TEXT = {
    "IMPROVED": "The whole interval is above zero: v2 is better on cases like these.",
    "REGRESSED": "The whole interval is below zero: v2 is worse on cases like these.",
    "INCONCLUSIVE": "The interval includes zero, so the data cannot tell v2 from v1. Do not report an improvement.",
}


def report_markdown(r: dict) -> str:
    one, many, rep, jr, ag, seg = (r["release_run"], r["twenty_runs"], r["repeated_runs"],
                                   r["judge_repeats"], r["agreement"], r["segments"])
    p, q = one["paired"], many["paired"]
    lines = [
        "# Reliability report: refund agent v2 against v1", "",
        f"> {r['label']}", "",
        f"Experiment `{r['experiment_version']}`, seed {r['seed']}, bootstrap {r['bootstrap']['reps']} resamples "
        f"(seed {r['bootstrap']['seed']}), {int(100 * r['bootstrap']['level'])}% intervals.", "",
        "## Decision", "",
        f"**{one['verdict']}** on the release run: v2 {pts(p['diff'])} points, interval {pts(p['lo'])} to {pts(p['hi'])}. "
        + DECISION_TEXT[one["verdict"]]
        + f" Detecting a 2-point gain with 80% power needs about {one['cases_needed_paired']:,} paired cases at this "
        f"run's discordance and {rep['cases_needed_at_mean']:,} at the 20-run average; this run has {p['n']}.", "",
        "## Release run (one run each, the same 100 cases)", "",
        "| Version | Passed | Pass rate | Wilson interval |", "|---|---:|---:|---|",
        f"| v1 | {round(p['a'] * p['n'])} | {pct(p['a'])} | {pct(one['wilson_a'][0])} to {pct(one['wilson_a'][1])} |",
        f"| v2 | {round(p['b'] * p['n'])} | {pct(p['b'])} | {pct(one['wilson_b'][0])} to {pct(one['wilson_b'][1])} |", "",
        "| Comparison | v2 minus v1 | Interval |", "|---|---:|---|",
        f"| Paired bootstrap | {pts(p['diff'])} | {pts(p['lo'])} to {pts(p['hi'])} |",
        f"| Unpaired bootstrap (pairing discarded, for contrast) | {pts(one['unpaired']['diff'])} | "
        f"{pts(one['unpaired']['lo'])} to {pts(one['unpaired']['hi'])} |", "",
        f"Discordant cases: {one['b_only']} only v2 passed, {one['a_only']} only v1 passed. "
        f"Exact McNemar p = {one['mcnemar_p']:.2f}.", "",
        "## Twenty runs each, averaged per case", "",
        f"v2 {pts(q['diff'])} points, paired interval {pts(q['lo'])} to {pts(q['hi'])}: **{many['verdict']}**. "
        f"Repeats shrink run-to-run noise. They do not add cases, so they say nothing new about requests unlike these 100.", "",
        "## Run-to-run spread with nothing changed", "",
        f"- SIMULATED agent: v1 passed {min(rep['counts']['v1'])} to {max(rep['counts']['v1'])} of {rep['n']} over "
        f"{len(rep['counts']['v1'])} runs; v2 passed {min(rep['counts']['v2'])} to {max(rep['counts']['v2'])}. "
        f"Single-run gaps (v2 minus v1) ranged from {min(rep['diffs']):+d} to {max(rep['diffs']):+d} cases, "
        f"and {min(rep['discordant'])} to {max(rep['discordant'])} cases were discordant per run pair.",
        f"- SIMULATED judge (Week 11, tool agent v2 unchanged): {min(jr['passes'])} to {max(jr['passes'])} of {jr['n']} "
        f"cases passed over {len(jr['passes'])} seeds, a spread of {jr['spread']} cases "
        f"({100 * jr['spread'] / jr['n']:.1f} points).", "",
        "## Judge and reviewer agreement", "",
        "**Labels are authored examples, not human review.** These numbers test the method, not the judge.", "",
        "| Raters | Cases | Agree | Cohen's kappa | Interval |", "|---|---:|---:|---:|---|",
        *[f"| {name} | {row['n']} | {row['agree']} | {row['kappa']:.3f} | {row['lo']:.3f} to {row['hi']:.3f} |"
          for name, row in ag["rows"].items()], "",
        f"Reviewer disagreements settled by the ledger (a forbidden tool call): {', '.join(ag['settled_by_ledger'])}. "
        f"Sent to the rubric owner: {', '.join(ag['for_rubric_owner'])}.", "",
        "## Segments (release run)", "",
        "| Slice | Cases | v1 | v2 | v1 interval | v2 interval |", "|---|---:|---:|---:|---|---|",
        *[f"| {s['slice']} | {s['n']} | {s['a']} | {s['b']} | {pct(s['wilson_a'][0])} to {pct(s['wilson_a'][1])} | "
          f"{pct(s['wilson_b'][0])} to {pct(s['wilson_b'][1])} |" for s in seg["slices"]], "",
        "Overall v1 pass rate on seven slices, each slice at its 20-run rate, under three mixes:", "",
        *[f"- {name}: {pct(rate)}" for name, rate in seg["overall"].items()], "",
        "## Caveats", "",
        "- Every pass and fail above is SIMULATED. The spread shows what sampling noise of this size does; "
        "it is not a measurement of any model's variation.",
        "- The 100 cases are the course's golden set. An interval covers requests like these cases, not traffic the set does not represent.",
        "- Intervals assume cases are independent. Near-duplicate cases make them too narrow.",
    ]
    return "\n".join(lines) + "\n"


def write_report(report: dict, out: Path) -> None:
    out.mkdir(parents=True, exist_ok=True)
    (out / "reliability-report.json").write_text(json.dumps(report, indent=2) + "\n")
    (out / "reliability-report.md").write_text(report_markdown(report))


print_segments(segment_report(exp))
report = build_report(exp)
write_report(report, HERE / "runs" / "week-15")
print(f"\n📝 wrote runs/week-15/reliability-report.md")
print("\n".join(report_markdown(report).splitlines()[6:9]))


`missing_order` drops from 7 of 7 to 5 of 7 with the same probability in both versions: noise, and one flipped case there is 14 points. With every slice rate held fixed, moving from baseline traffic to the clearance-sale mix lowers v1's overall rate from 85.8% to 76.5%, because final-sale requests go from 5% to 40% of traffic. The report's decision line repeats the release-run verdict with the cases a 2-point claim would need.


---
## 🧪 Your Own Experiment

Change `case_share` or a slice's probability in the copy below and rerun. At `case_share = 0` every run is independent and pairing helps little. Near 1 the agent is almost deterministic and the discordant cases shrink.


In [ ]:
# ============================================================================
# YOUR OWN EXPERIMENT: Edit the copy, not the file
# ============================================================================
mine = copy.deepcopy(exp)
mine["case_share"] = 0.0                                       # try 0.0, 0.7, 0.95
mine["pass_probability"]["v2"]["window_boundary"] = 0.75      # v2's gain lives in this slice and amount_boundary
print_compare(compare(mine))


---
## 🐛 Drill 1: Pairing by Position

v2's results come back as a list in completion order, as parallel workers return them, and the comparison matches the two lists by position. Both lists hold the same 100 case IDs, so `paired_difference` raises nothing.


In [ ]:
# ============================================================================
# DRILL 1: Results matched by position instead of case ID
# ============================================================================
rows_v1 = [{"case_id": i, "passed": v1_release[i]} for i in sorted(v1_release)]
rows_v2 = [{"case_id": i, "passed": v2_release[i]} for i in sorted(v2_release)]
random.Random(7).shuffle(rows_v2)                              # completion order

by_position = {r1["case_id"]: r2["passed"] for r1, r2 in zip(rows_v1, rows_v2)}
broken = paired_difference(v1_release, by_position, 10000, 15)
print(f"❌ position-paired interval: {pts(broken['lo'])} to {pts(broken['hi'])}, "
      f"discordant {discordant(v1_release, by_position)}")

by_case = {r["case_id"]: r["passed"] for r in rows_v2}
fixed = paired_difference(v1_release, by_case, 10000, 15)
print(f"✅ case-ID-paired interval: {pts(fixed['lo'])} to {pts(fixed['hi'])}, "
      f"discordant {discordant(v1_release, by_case)}")


---
## 🐛 Drill 2: Certainty from 30 of 30

A report puts a Wald interval (the estimate plus and minus 1.96 standard errors) on each critical slice. For 30 passes in 30 cases the standard error is zero.


In [ ]:
# ============================================================================
# DRILL 2: Wald against Wilson at the edges
# ============================================================================
def wald_interval(k, n, level=0.95):
    z, p = z_value(level), k / n
    half = z * math.sqrt(p * (1 - p) / n)
    return max(0.0, p - half), min(1.0, p + half)


for k, n in ((30, 30), (1, 6)):
    lo, hi = wald_interval(k, n)
    print(f"❌ Wald {k}/{n}: {pct(lo)} to {pct(hi)}")
    lo, hi = wilson(k, n)
    print(f"✅ Wilson {k}/{n}: {pct(lo)} to {pct(hi)}")


---
## 📝 Summary

In this notebook, we learned:

### 1. Repeated runs
- With nothing changed, v1 moved 12 cases across 20 runs and the judge moved 6 of 30. A 2-point difference between two runs is inside that noise.

### 2. Intervals and the bootstrap
- The bootstrap resamples cases with replacement and reads the spread. Wilson does the same job for one pass rate by formula, and 30 of 30 is 88.6% to 100%, not 100%.

### 3. Paired comparisons
- Only the 8 discordant cases carry evidence about v2. The paired interval is −3.0 to +8.0: INCONCLUSIVE. A 2-point gain needs 1,562 to 3,142 paired cases, depending on the discordance.

### 4. Judge and reviewer agreement
- Kappa from 26 labels spans chance to near-perfect. Two reviewers set the ceiling, and their splits are either ledger facts or rubric questions.

### 5. Segments and the report
- Small slices are mostly noise, and a mix shift can drop the overall rate 9 points with every slice unchanged.

### Next Steps
Week 16's notebook, `week-16-distributed-evaluation-capstone-handson.ipynb`, runs these comparisons as jobs on a shared multi-tenant worker queue, with budgets, retries and the release decision's audit trail.
